## Sentence-Embedding-V3-large Baseline

Dieses Modell wird von OpenAI über eine API bereitgestellt und zeigt starke Ergebnisse in diversen Retrieval-Benchmarks. Das Modell ist nicht Open Source und kann nicht mittels eines Adapters angepasst werden, dient im Rahmen der Arbeit jedoch als starke Baseline.

In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, Iterable, List
import os

import numpy as np
from openai import OpenAI


# Eingabedateien und Ausgabepfad
CORPUS_PATH = "datasets/GerManualDPR/full_corpus.json"
TEST_PATH = "datasets/GerManualDPR/manual/test_dataset.json"
OUTPUT_PATH = "results/predictions/baseline/text-embedding-3-large_2/Information-Retrieval_evaluation_myEvaluator_predictions_cosine.jsonl"
os.environ["OPENAI_API_KEY"] = "Hier eigenen OPENAI_API_KEY einfügen"

# Modell- und Retrieval-Konfiguration
MODEL_NAME = "text-embedding-3-large"
TOP_K = 100
BATCH_SIZE = 64


def load_json(path: str) -> List[Dict[str, Any]]:
    """Lädt eine JSON-Datei und gibt deren Inhalt als Liste von Dictionaries zurück."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def batched(items: List[str], batch_size: int) -> Iterable[List[str]]:
    """Teilt eine Liste in aufeinanderfolgende Batches fester Größe auf."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def normalize(embeddings: np.ndarray) -> np.ndarray:
    """Normiert Embeddings auf Länge 1, damit Kosinusähnlichkeit per Skalarprodukt berechnet werden kann."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / np.clip(norms, 1e-12, None)


def get_embeddings(client: OpenAI, texts: List[str], model: str, batch_size: int) -> np.ndarray:
    """Erzeugt Embeddings für alle Texte in Batches über die OpenAI-API."""
    vectors = []

    for batch in batched(texts, batch_size):
        # Leere Strings werden durch ein Leerzeichen ersetzt, damit die API keine ungültigen Eingaben erhält.
        batch = [text if text.strip() else " " for text in batch]
        response = client.embeddings.create(model=model, input=batch)
        vectors.extend(item.embedding for item in response.data)

    return np.asarray(vectors, dtype=np.float32)


def top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    """Liefert die Indizes der k höchsten Scores in absteigender Reihenfolge zurück."""
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)

    # argpartition ist effizienter als vollständiges Sortieren, wenn nur die Top-k benötigt werden.
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]


def main() -> None:
    client = OpenAI()

    print("Lade Daten ...")
    corpus_rows = load_json(CORPUS_PATH)
    test_rows = load_json(TEST_PATH)

    # Extrahiere Korpus-IDs und Korpustexte
    corpus_ids = [row["chunk_id"] for row in corpus_rows]
    corpus_texts = [row["positive"] for row in corpus_rows]

    # Extrahiere Query-Informationen aus dem Testdatensatz
    query_ids = [row["id"] for row in test_rows]
    query_texts = [row["anchor"] for row in test_rows]
    gold_ids = [row["chunk_id"] for row in test_rows]

    print(f"Korpus-Chunks: {len(corpus_rows)}")
    print(f"Test-Queries: {len(test_rows)}")

    print("Erzeuge Korpus-Embeddings ...")
    corpus_emb = normalize(get_embeddings(client, corpus_texts, MODEL_NAME, BATCH_SIZE))

    print("Erzeuge Query-Embeddings ...")
    query_emb = normalize(get_embeddings(client, query_texts, MODEL_NAME, BATCH_SIZE))

    # Falls der Ausgabeordner noch nicht existiert, wird er automatisch angelegt.
    Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

    print(f"Berechne Top-{TOP_K}-Predictions ...")
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for qid, query, gold_id, q_vec in zip(query_ids, query_texts, gold_ids, query_emb):
            # Da alle Vektoren normiert sind, entspricht das Skalarprodukt der Kosinusähnlichkeit.
            scores = corpus_emb @ q_vec
            top_idx = top_k_indices(scores, TOP_K)

            results = [
                {"corpus_id": corpus_ids[i], "score": float(scores[i])}
                for i in top_idx
            ]

            record = {
                "query_id": qid,
                "query": query,
                "gold_chunk_id": gold_id,
                "results": results,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("Fertig.")


if __name__ == "__main__":
    main()